In [8]:
import os
import shutil
import zipfile
import kagglehub
from pathlib import Path
from zipfile import ZipFile
from src.TALOS import logger
from src.TALOS.utils import *
import urllib.request as request
from src.TALOS.utils.common import read_yaml,create_directories,get_size
from src.TALOS.constants import CONFIG_FILE_PATH,PARAMS_FILE_PATH,SCHEMA_FILE_PATH

In [1]:
import os
%pwd

'/Users/gojuruakshith/Tactical-Aerospace-Localization-Objective-Scoring-TALOS-/research'

In [6]:
cd ..

/Users/gojuruakshith/Tactical-Aerospace-Localization-Objective-Scoring-TALOS-


In [64]:
cd Tactical-Aerospace-Localization-Objective-Scoring-TALOS-

/Users/gojuruakshith/Tactical-Aerospace-Localization-Objective-Scoring-TALOS-


In [9]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [10]:
class ConfigurationManager:
    def __init__(self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )


In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Fetching {self.config.source_URL} via kagglehub...")
            downloaded_path = kagglehub.dataset_download(self.config.source_URL)
            os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
            if os.path.exists(downloaded_path):
                shutil.move(downloaded_path, self.config.local_data_file)
                logger.info(f"Dataset moved to: {self.config.local_data_file}")
        else:
            logger.info(f"Data already present at: {self.config.local_data_file}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        if self.config.local_data_file != unzip_path:
            if os.path.exists(self.config.local_data_file):
                logger.info(f"Syncing data to {unzip_path}")
                shutil.copytree(self.config.local_data_file, unzip_path, dirs_exist_ok=True)

In [13]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-02-17 18:05:36,162: INFO: common: yaml_path: config/config.yaml loaded successfully]
[2026-02-17 18:05:36,164: INFO: common: yaml_path: params.yaml loaded successfully]
[2026-02-17 18:05:36,165: INFO: common: yaml_path: schema.yaml loaded successfully]
hi
[2026-02-17 18:05:36,166: INFO: common: created directory at: artifacts]
hi
[2026-02-17 18:05:36,167: INFO: common: created directory at: artifacts/data_ingestion]
[2026-02-17 18:05:36,167: INFO: 3082532317: Data already present at: artifacts/data_ingestion/data.zip]
[2026-02-17 18:05:36,168: INFO: 3082532317: Syncing data to artifacts/data_ingestion]
